In [ ]:
#@title Cell 41.1 — Overview and BMC-N model
from IPython.display import Markdown, display

display(Markdown(r"""
# NB41 — BMC-N Bayesian calibration and pathogen-out validation (v0.1)

**Objective:** Apply **Bayesian Model C — Normal (BMC-N)** to all 26 antibiotics, using the exact training-only cross-fitted predictions from NB40, and evaluate on the original 50,460 independently held-out Model C predictions.

For antibiotic $a$ and pathogen–antibiotic record $i$, let $r_i$ be the **Model C** predicted log₂(MIC) and $y_i$ the observed log₂(MIC). We fit:

$$ y_i\mid r_i,b_a,\sigma_a \sim \mathcal N(r_i+b_a,\sigma_a^2),$$

with independent priors $b_a\sim\mathcal N(0,1^2)$ and $\sigma_a\sim\operatorname{Exponential}(0.5)$ (rate parameter). Each of the **five outer folds × 26 antibiotics** has its own training-only Bayesian fit. No chromosomal MIC score is used.

**Method:** Integrate out the normal bias analytically conditional on $\sigma_a$; evaluate the one-dimensional posterior of $\sigma_a$ on a fine logarithmic grid. Compute posterior predictive 2.5%, 50%, 97.5% quantiles from the exact *normal-mixture CDF* by numerical inversion. This avoids MCMC convergence considerations or random simulation noise in reported intervals.

**Independent evaluation:** BMC parameters for outer fold $k$ use only NB40's inner out-of-fold Model C residuals for outer-training pathogens; the original outer test patients' observed MICs are reserved for evaluation. The five calibration datasets overlap and must not be pooled as independent observations. No common resistance breakpoint is assumed across antibiotics.

**Figures:** (1) 95% coverage across all 26 antibiotics, (2) observed MICs above upper 95% predictive limit by antibiotic, (3) individual BioSamples' intervals for four focus antibiotics (ceftazidime, cefepime, ciprofloxacin, meropenem). The third plot is deliberately limited to 25 high-underprediction cases per antibiotic for readability.

**Inputs:** Two NB40 output files only. **Outputs:** `RefMIC/03_Results/41_BMC_N_Pathogen_Out_Validation`. Run all cells **41.1–41.8** in order. No original source files are modified.

**Caution:** The original Model C kernels use cohort-wide genomic information (transductive construction); model settings were selected within each outer training fold. NB41's evaluation is held-out **for MIC outcomes**, under those same Model C design choices.
"""))


In [ ]:
#@title Cell 41.2 — Locate NB40 outputs automatically
import os, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.special import logsumexp, ndtr
from scipy.optimize import brentq
import matplotlib.pyplot as plt
from IPython.display import display

try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive')
except ImportError:
    pass

ROOT=Path(os.environ.get('BMC_PROJECT_ROOT','/content/drive/MyDrive/RefMIC'))
if not ROOT.is_dir():
    raise FileNotFoundError(f'RefMIC project folder not found at {ROOT}; set BMC_PROJECT_ROOT if it is elsewhere.')
INPUT_DIR=ROOT/'03_Results'/'40_BMC_Training_Only_Calibration'
OUT=ROOT/'03_Results'/'41_BMC_N_Pathogen_Out_Validation'
OUT.mkdir(parents=True,exist_ok=True)

CALIB_NAME='40_BMC_training_only_inner_predictions_all_outer_folds.csv.gz'
TEST_NAME='40_BMC_independent_outer_test_predictions.csv.gz'

def get_file(name):
    direct=INPUT_DIR/name
    if direct.is_file(): return direct
    searchroot=Path('/content/drive') if Path('/content/drive').is_dir() else ROOT
    matches=[]
    for directory,dirs,files in os.walk(searchroot):
        dirs[:]=[x for x in dirs if x not in {'.Trashes','.Trash-1000','.git','__pycache__','node_modules'}]
        if name in files: matches.append(Path(directory)/name)
    if not matches:
        raise FileNotFoundError(f'Could not find {name} under {searchroot}. Please run NB40 Cell 40.7.')
    if len(matches)>1:
        def sha(p):
            h=hashlib.sha256()
            with open(p,'rb') as f:
                for buf in iter(lambda:f.read(4*1024*1024),b''): h.update(buf)
            return h.hexdigest()
        if len({sha(p) for p in matches})!=1:
            raise ValueError(f'Different copies of {name} found; unable to choose:\n'+ '\n'.join(map(str,matches)))
        print('Multiple byte-identical copies found for',name)
    return sorted(matches,key=lambda p:(0 if ROOT in p.parents else 1,len(p.parts),str(p)))[0]

CALIB_PATH=get_file(CALIB_NAME)
TEST_PATH=get_file(TEST_NAME)
print('Training-only calibration:',CALIB_PATH)
print('Independent held-out test:',TEST_PATH)
print('NB41 results:',OUT)


In [ ]:
#@title Cell 41.3 — Load inputs and verify fold separation
calib=pd.read_csv(CALIB_PATH,compression='gzip')
outer_test=pd.read_csv(TEST_PATH,compression='gzip')

calib_cols={'outer_fold_for_calibration','inner_fold','interaction_row','biosample','antibiotic','observed_log2_mic','inner_oof_predicted_log2_mic','training_residual_log2'}
test_cols={'outer_fold','interaction_row','biosample','antibiotic','observed_log2_mic','model_c_outer_test_predicted_log2_mic'}
for name,tab,needed in [('calibration',calib,calib_cols),('outer test',outer_test,test_cols)]:
    if not needed.issubset(tab.columns):
        raise ValueError(f'{name}: missing columns {sorted(needed-set(tab.columns))}')

if len(calib)!=201840 or len(outer_test)!=50460:
    raise ValueError('NB40 data sizes do not match 201,840 training records and 50,460 outer test records.')
if outer_test.biosample.nunique()!=9058 or outer_test.antibiotic.nunique()!=26:
    raise ValueError('Unexpected number of BioSamples or antibiotics.')
if outer_test.interaction_row.duplicated().any() or outer_test.duplicated(['biosample','antibiotic']).any():
    raise ValueError('Duplicate independent Model C interactions.')
if (outer_test.groupby('biosample').outer_fold.nunique()!=1).any():
    raise ValueError('Outer-test BioSample appears in more than one fold.')
if set(outer_test.outer_fold.unique())!=set(range(1,6)) or set(calib.outer_fold_for_calibration.unique())!=set(range(1,6)):
    raise ValueError('Expected exactly five matching outer folds.')
if calib.duplicated(['outer_fold_for_calibration','interaction_row']).any():
    raise ValueError('Duplicate calibration records within an outer fold.')
if (calib.groupby(['outer_fold_for_calibration','biosample']).inner_fold.nunique()!=1).any():
    raise ValueError('A calibration BioSample appears in more than one inner fold.')

original=outer_test.set_index('interaction_row')[['biosample','antibiotic','observed_log2_mic','outer_fold']]
for fold in range(1,6):
    cal=calib.loc[calib.outer_fold_for_calibration.eq(fold)]
    test=outer_test.loc[outer_test.outer_fold.eq(fold)]
    if len(cal)!=len(outer_test)-len(test):raise ValueError(f'Fold {fold} wrong calibration size.')
    if set(cal.interaction_row)!=set(outer_test.interaction_row)-set(test.interaction_row):
        raise ValueError(f'Fold {fold} calibration interactions do not match outer training set.')
    if set(cal.biosample)&set(test.biosample):raise ValueError(f'Fold {fold} test/calibration BioSample overlap.')
    check=original.loc[cal.interaction_row.to_numpy()]
    if not np.array_equal(check.biosample.astype(str).to_numpy(),cal.biosample.astype(str).to_numpy()):
        raise ValueError(f'Fold {fold} BioSample mismatch.')
    if not np.array_equal(check.antibiotic.astype(str).to_numpy(),cal.antibiotic.astype(str).to_numpy()):
        raise ValueError(f'Fold {fold} antibiotic mismatch.')
    if not np.allclose(check.observed_log2_mic.to_numpy(),cal.observed_log2_mic.to_numpy(),atol=1e-9,rtol=0):
        raise ValueError(f'Fold {fold} observed MIC mismatch.')

for tab,nums in [(calib,['observed_log2_mic','inner_oof_predicted_log2_mic','training_residual_log2']),
                 (outer_test,['observed_log2_mic','model_c_outer_test_predicted_log2_mic'])]:
    if not np.isfinite(tab[nums].to_numpy(dtype=float)).all():raise ValueError('Non-finite MIC values/predictions.')
if not np.allclose(calib.training_residual_log2,calib.observed_log2_mic-calib.inner_oof_predicted_log2_mic,rtol=0,atol=1e-8):
    raise ValueError('NB40 residuals are inconsistent with observed minus predicted MIC.')

print(f'Validated: {len(calib):,} overlapping training-only records across five outer fits.')
print(f'Validated: {len(outer_test):,} independent held-out records; {outer_test.biosample.nunique():,} BioSamples; 26 antibiotics.')
print('All five folds: calibration/test BioSamples disjoint and observation values identical.')
display(outer_test.groupby('outer_fold',as_index=False).agg(test_rows=('interaction_row','size'),test_pathogens=('biosample','nunique')))


In [ ]:
#@title Cell 41.4 — Fit Bayesian Normal posteriors using training data only
# Exact marginal integration of b | sigma, followed by 1D numerical integration of sigma.
# Priors: b~N(0,1), sigma~Exp(rate=0.5). log(sigma) grid includes Jacobian.
SIGMA_GRID=np.exp(np.linspace(np.log(0.01),np.log(16.0),4501))
LOG_SIGMA=np.log(SIGMA_GRID)
SIGMA2=SIGMA_GRID**2
PRIOR_RATE=0.5

def mixture_quantiles(quantiles, component_mean, component_sd, weights):
    """Numerically invert an exact normal-mixture CDF (one scalar per requested percentile)."""
    component_mean=np.asarray(component_mean,dtype=float)
    component_sd=np.asarray(component_sd,dtype=float)
    weights=np.asarray(weights,dtype=float)
    lo=float(np.min(component_mean-10*component_sd))
    hi=float(np.max(component_mean+10*component_sd))
    def cdf(x):return float(np.dot(weights,ndtr((x-component_mean)/component_sd)))
    return [float(brentq(lambda x:cdf(x)-float(q),lo,hi,xtol=1e-10)) for q in quantiles]

def weighted_grid_quantile(x,w,q):
    return float(np.interp(q,np.cumsum(w),x))

def fit_bmc_n(errors):
    e=np.asarray(errors,dtype=np.float64)
    n=len(e)
    if n<15 or not np.isfinite(e).all():raise ValueError('Insufficient/non-finite calibration residuals.')
    mean=float(e.mean());sse=float(np.sum((e-mean)**2))
    # Covariance after integrating b: sigma^2 I + 1 1^T, because Var(b)=1.
    # log p(sigma|e) wrt log(sigma) = log p(e|sigma) + log p(sigma) + log(sigma) + constant.
    log_density=( -(n-1)*LOG_SIGMA-0.5*np.log(SIGMA2+n)
                 -sse/(2*SIGMA2)-n*mean**2/(2*(SIGMA2+n))
                 -PRIOR_RATE*SIGMA_GRID+LOG_SIGMA )
    w=np.exp(log_density-logsumexp(log_density))
    edge_mass=float(w[:5].sum()+w[-5:].sum())
    if edge_mass>1e-8:
        raise ValueError(f'Posterior sigma grid truncated: endpoint mass={edge_mass:.3g}; enlarge grid.')
    # b | sigma,e ~ N( n*mean/(n+sigma²), sigma²/(n+sigma²)).
    b_mean_condition=n*mean/(n+SIGMA2)
    b_sd_condition=np.sqrt(SIGMA2/(n+SIGMA2))
    predictive_sd=np.sqrt(SIGMA2+b_sd_condition**2)
    p025,p50,p975=mixture_quantiles([.025,.5,.975],b_mean_condition,predictive_sd,w)
    b025,b975=mixture_quantiles([.025,.975],b_mean_condition,b_sd_condition,w)
    return {
        'n_calibration':n,'mean_training_residual_log2':mean,
        'posterior_mean_b_log2':float(np.dot(w,b_mean_condition)),
        'posterior_b_2p5pct':b025,'posterior_b_97p5pct':b975,
        'posterior_mean_sigma_log2':float(np.dot(w,SIGMA_GRID)),
        'posterior_median_sigma_log2':weighted_grid_quantile(SIGMA_GRID,w,.5),
        'posterior_sigma_2p5pct':weighted_grid_quantile(SIGMA_GRID,w,.025),
        'posterior_sigma_97p5pct':weighted_grid_quantile(SIGMA_GRID,w,.975),
        'predictive_residual_lower95_log2':p025,
        'predictive_residual_median_log2':p50,
        'predictive_residual_upper95_log2':p975,
        'sigma_grid_endpoint_mass':edge_mass,
    }

post_rows=[]
for (fold,antibiotic),part in calib.groupby(['outer_fold_for_calibration','antibiotic'],sort=True):
    post_rows.append({'outer_fold':int(fold),'antibiotic':str(antibiotic),
                      **fit_bmc_n(part.training_residual_log2.to_numpy())})
posterior_summary=pd.DataFrame(post_rows).sort_values(['outer_fold','antibiotic']).reset_index(drop=True)
if len(posterior_summary)!=5*26:
    raise ValueError(f'Expected 130 antibiotic-fold fits; got {len(posterior_summary)}.')
assert posterior_summary.sigma_grid_endpoint_mass.max()<=1e-8
print('Fit complete:',len(posterior_summary),'separate training-only antibiotic × outer-fold Bayesian fits.')
print('Numerical integration: 1D log(sigma) quadrature; no MCMC required.')
print('Maximum posterior endpoint mass:',f'{posterior_summary.sigma_grid_endpoint_mass.max():.2g}')
display(posterior_summary[['outer_fold','antibiotic','n_calibration',
                           'posterior_mean_b_log2','posterior_median_sigma_log2',
                           'predictive_residual_lower95_log2',
                           'predictive_residual_upper95_log2']].head(10).round(4))


In [ ]:
#@title Cell 41.5 — Evaluate independent outer-test predictions and summarise 26 antibiotics
shifts=posterior_summary[['outer_fold','antibiotic','predictive_residual_lower95_log2',
                          'predictive_residual_median_log2','predictive_residual_upper95_log2']]
bmc=outer_test.merge(shifts,on=['outer_fold','antibiotic'],how='left',validate='many_to_one',indicator=True)
if len(bmc)!=50460 or not bmc._merge.eq('both').all():
    raise ValueError('Some outer test predictions lack matching Bayesian parameters.')
bmc=bmc.drop(columns=['_merge'])
r=bmc.model_c_outer_test_predicted_log2_mic.to_numpy()
bmc['bmc_n_lower95_log2']=r+bmc.predictive_residual_lower95_log2
bmc['bmc_n_median_log2']=r+bmc.predictive_residual_median_log2
bmc['bmc_n_upper95_log2']=r+bmc.predictive_residual_upper95_log2
bmc=bmc.drop(columns=['predictive_residual_lower95_log2','predictive_residual_median_log2','predictive_residual_upper95_log2'])
bmc['model_c_residual_log2']=bmc.observed_log2_mic-bmc.model_c_outer_test_predicted_log2_mic
bmc['bmc_n_residual_log2']=bmc.observed_log2_mic-bmc.bmc_n_median_log2
bmc['covered_95pct']=(bmc.observed_log2_mic>=bmc.bmc_n_lower95_log2)&(bmc.observed_log2_mic<=bmc.bmc_n_upper95_log2)
bmc['above_upper95']=bmc.observed_log2_mic>bmc.bmc_n_upper95_log2
bmc['below_lower95']=bmc.observed_log2_mic<bmc.bmc_n_lower95_log2
bmc['distance_above_upper_log2']=np.maximum(0,bmc.observed_log2_mic-bmc.bmc_n_upper95_log2)
bmc['model_c_underprediction_ge2']=bmc.model_c_residual_log2>=2
bmc['model_c_underprediction_ge4']=bmc.model_c_residual_log2>=4
bmc['bmc_n_underprediction_ge2']=bmc.bmc_n_residual_log2>=2
bmc['bmc_n_underprediction_ge4']=bmc.bmc_n_residual_log2>=4
bmc['interval_width_log2']=bmc.bmc_n_upper95_log2-bmc.bmc_n_lower95_log2
if (bmc.interval_width_log2<=0).any():raise ValueError('Invalid Bayesian predictive intervals.')

def summarise(tab):
    mc=tab.model_c_residual_log2.to_numpy()
    bn=tab.bmc_n_residual_log2.to_numpy()
    return pd.Series({
        'n_test':len(tab),'n_pathogens':tab.biosample.nunique(),
        'model_c_MAE_log2':np.abs(mc).mean(),
        'bmc_n_MAE_log2':np.abs(bn).mean(),
        'model_c_RMSE_log2':np.sqrt(np.mean(mc**2)),
        'bmc_n_RMSE_log2':np.sqrt(np.mean(bn**2)),
        'interval_95pct_coverage':tab.covered_95pct.mean(),
        'n_above_upper95':int(tab.above_upper95.sum()),
        'n_below_lower95':int(tab.below_lower95.sum()),
        'pct_above_upper95':100*tab.above_upper95.mean(),
        'mean_interval_width_log2':tab.interval_width_log2.mean(),
        'n_model_c_underprediction_ge2':int(tab.model_c_underprediction_ge2.sum()),
        'n_model_c_underprediction_ge4':int(tab.model_c_underprediction_ge4.sum()),
        'n_bmc_n_underprediction_ge2':int(tab.bmc_n_underprediction_ge2.sum()),
        'n_bmc_n_underprediction_ge4':int(tab.bmc_n_underprediction_ge4.sum()),
    })

overall_summary=summarise(bmc).to_frame().T
antibiotic_summary=pd.DataFrame([{'antibiotic':name,**summarise(part).to_dict()}
                                 for name,part in bmc.groupby('antibiotic',sort=False)])
# Ensure numerical tables remain correctly typed and sorted by upper-limit misses.
antibiotic_summary=antibiotic_summary.sort_values(['n_above_upper95','interval_95pct_coverage'],ascending=[False,True]).reset_index(drop=True)
print('Overall held-out performance (50,460 observations):')
display(overall_summary.round(4))
print('Antibiotic-specific held-out performance (26 antibiotics):')
display(antibiotic_summary.round(4))
print('Reminder: coverage is a predictive interval metric; no universal resistance threshold is used.')


In [ ]:
#@title Cell 41.6 — Figures 1 and 2: coverage and upper-tail misses across 26 antibiotics
plot_table=antibiotic_summary.sort_values('interval_95pct_coverage',ascending=True)
fig,ax=plt.subplots(figsize=(11,11))
pos=np.arange(len(plot_table))
ax.barh(pos,100*plot_table.interval_95pct_coverage,color='C0',alpha=.82)
ax.axvline(95,color='black',ls='--',lw=1.5,label='Nominal 95% coverage')
ax.set_yticks(pos,plot_table.antibiotic)
ax.set_xlim(0,103)
ax.set_xlabel('Held-out observations within 95% BMC-N predictive interval (%)')
ax.set_title('Figure 1 — BMC-N coverage across 26 antibiotics')
ax.legend(loc='lower right')
ax.grid(axis='x',alpha=.2)
ax.set_axisbelow(True)
for y0,val,n in zip(pos,plot_table.interval_95pct_coverage,plot_table.n_test):
    ax.text(min(100*val+0.7,100.1),y0,f'n={int(n)}',va='center',fontsize=8)
fig.tight_layout()
fig.savefig(OUT/'41_Figure1_26_Antibiotic_Coverage.png',dpi=180,bbox_inches='tight')
plt.show()

miss_table=antibiotic_summary.sort_values('n_above_upper95',ascending=True)
fig,ax=plt.subplots(figsize=(11,11))
pos=np.arange(len(miss_table))
ax.barh(pos,miss_table.n_above_upper95,color='C1',alpha=.85)
ax.set_yticks(pos,miss_table.antibiotic)
ax.set_xlabel('Count: observed log₂(MIC) above BMC-N upper 95% predictive limit')
ax.set_title('Figure 2 — Upper-tail misses among independent held-out predictions')
ax.grid(axis='x',alpha=.2)
ax.set_axisbelow(True)
maxmiss=int(miss_table.n_above_upper95.max())
ax.set_xlim(0,max(5,maxmiss*1.25+1))
for y0,count,pct in zip(pos,miss_table.n_above_upper95,miss_table.pct_above_upper95):
    ax.text(int(count)+max(0.2,maxmiss*.012),y0,f'{int(count)} ({pct:.1f}%)',va='center',fontsize=8)
fig.tight_layout()
fig.savefig(OUT/'41_Figure2_26_Antibiotic_Upper_Misses.png',dpi=180,bbox_inches='tight')
plt.show()


In [ ]:
#@title Cell 41.7 — Figure 3: individual held-out pathogens for four focus antibiotics
FOCUS_ANTIBIOTICS=['ceftazidime','cefepime','ciprofloxacin','meropenem']
FOCUS_N=25

for ab in FOCUS_ANTIBIOTICS:
    d=bmc.loc[bmc.antibiotic.eq(ab)].copy()
    if d.empty: raise ValueError(f'No independent held-out observations for {ab}.')
    # Highlight the most severe upper-tail underpredictions. When above-upper misses
    # are fewer than 25, also show the next largest positive Model C residuals.
    d=d.sort_values(['distance_above_upper_log2','model_c_residual_log2'],ascending=[False,False]).head(FOCUS_N)
    yrows=np.arange(len(d))
    fig,ax=plt.subplots(figsize=(12,max(6,0.35*len(d)+2)))
    ax.hlines(yrows,d.bmc_n_lower95_log2,d.bmc_n_upper95_log2,color='C0',lw=2.5,
              label='BMC-N 95% predictive interval')
    ax.scatter(d.bmc_n_median_log2,yrows,c='C0',marker='o',s=27,zorder=3,label='BMC-N predictive median')
    ax.scatter(d.observed_log2_mic,yrows,c='C3',marker='x',s=47,lw=1.8,zorder=4,label='Observed log₂(MIC)')
    ax.set_yticks(yrows,d.biosample)
    ax.invert_yaxis()
    ax.set_xlabel('log₂(MIC), MIC in mg/L')
    ax.set_ylabel('BioSample')
    ax.set_title(f'Figure 3 — {ab}: {len(d)} largest underprediction cases (held-out)')
    ax.grid(axis='x',alpha=.23)
    ax.set_axisbelow(True)
    ax.legend(loc='lower right',fontsize=9)
    fig.tight_layout()
    fname=f'41_Figure3_Intervals_{ab.replace("-","_").replace(" ","_")}.png'
    fig.savefig(OUT/fname,dpi=180,bbox_inches='tight')
    plt.show()
    print(ab,':',int(d.above_upper95.sum()),'of plotted 25 above the upper 95% limit;',
          int(bmc.loc[bmc.antibiotic.eq(ab),'above_upper95'].sum()),'above among all held-out observations.')


In [ ]:
#@title Cell 41.8 — Save results and final validation audit
PRED_OUT=OUT/'41_BMC_N_independent_outer_test_predictions.csv.gz'
POST_OUT=OUT/'41_BMC_N_training_only_posterior_by_antibiotic_and_fold.csv'
ANT_OUT=OUT/'41_BMC_N_26_antibiotic_validation_summary.csv'
ALL_OUT=OUT/'41_BMC_N_overall_validation_summary.csv'
AUDIT_OUT=OUT/'41_BMC_N_validation_audit.txt'

bmc.to_csv(PRED_OUT,index=False,compression='gzip')
posterior_summary.to_csv(POST_OUT,index=False)
antibiotic_summary.to_csv(ANT_OUT,index=False)
overall_summary.to_csv(ALL_OUT,index=False)

checks={
    'total_held_out_rows':len(bmc),
    'held_out_biosamples':int(bmc.biosample.nunique()),
    'antibiotics':int(bmc.antibiotic.nunique()),
    'model_fits':len(posterior_summary),
    'calibration_records_across_five_overlapping_outer_training_folds':len(calib),
    'distinct_independent_held_out_records':int(bmc.interaction_row.nunique()),
    'all_folds_pathogen_disjoint':True,
    'maximum_sigma_grid_endpoint_mass':float(posterior_summary.sigma_grid_endpoint_mass.max()),
    'coverage':float(bmc.covered_95pct.mean()),
    'n_upper_misses':int(bmc.above_upper95.sum()),
    'n_lower_misses':int(bmc.below_lower95.sum()),
}
if (checks['total_held_out_rows'],checks['held_out_biosamples'],checks['antibiotics'],checks['model_fits'])!=(50460,9058,26,130):
    raise ValueError('Final BMC-N data size checks failed.')
AUDIT_OUT.write_text('\n'.join(f'{k}: {v}' for k,v in checks.items())+'\n',encoding='utf-8')
print('NB41 completed successfully. All reported BMC-N metrics use independent held-out MIC observations.')
for f in [PRED_OUT,POST_OUT,ANT_OUT,ALL_OUT,AUDIT_OUT]:print('Saved:',f)
print('Saved six PNG figures in:',OUT)
print('Next: compare BMC-N with BMC-T and BMC-AT under identical outer folds and plotting rules.')
